# RAG パイプライン検証環境

詳細な処理ステップとトークン使用量を可視化した実験環境

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import display, HTML, Markdown
import json

# backend をパスに追加
backend_path = Path(".").resolve().parent.parent / "backend"
sys.path.insert(0, str(backend_path))

from app.rag.embedder import Embedder
from app.rag.retriever import Retriever
from app.rag.generator import Generator
from app.rag.pipeline import RAGPipeline
from app.config import config

print(f"Backend path: {backend_path}")
print(f"Python path: {sys.path[:2]}")

## 1. 初期化

In [ ]:
# API key check
print("="*60)
print("RAG Pipeline 初期化")
print("="*60)

if not config.GEMINI_API_KEY:
    print("⚠️  GEMINI_API_KEY が設定されていません")
else:
    print("✓ GEMINI_API_KEY: 設定済み")

# Chroma DB check
db_path = Path(config.CHROMA_DB_PATH)
if not db_path.exists():
    print(f"❌ Chroma DB が見つかりません: {config.CHROMA_DB_PATH}")
    print(f"   実行: python rebuild_index.py")
    sys.exit(1)
else:
    print(f"✓ Chroma DB: {config.CHROMA_DB_PATH}")

print(f"✓ Embedding モデル: {config.EMBEDDING_MODEL_NAME}")
print(f"✓ Top-K: {config.CHROMA_TOP_K}")
print()

# Initialize RAG pipeline
print("初期化中...")
embedder = Embedder(model_name=config.EMBEDDING_MODEL_NAME)
retriever = Retriever(db_path=config.CHROMA_DB_PATH)
retriever.get_collection()
generator = Generator(api_key=config.GEMINI_API_KEY)
pipeline = RAGPipeline(embedder, retriever, generator)

print("✓ RAG パイプライン: 準備完了")
print("="*60)

## 2. クエリ実行（詳細情報表示）

In [ ]:
def format_metric_display(metrics):
    """メトリクス情報をフォーマット表示"""
    html = f"""
    <div style="background-color: #f5f5f5; padding: 15px; border-radius: 5px; margin: 10px 0;">
    <h3>⏱️ 処理時間</h3>
    <table style="width: 100%;">
        <tr><td>エンベディング</td><td style="text-align: right;"><b>{metrics['embedding_time']:.3f}s</b></td></tr>
        <tr><td>検索</td><td style="text-align: right;"><b>{metrics['search_time']:.3f}s</b></td></tr>
        <tr><td>生成</td><td style="text-align: right;"><b>{metrics['generation_time']:.3f}s</b></td></tr>
        <tr style="border-top: 1px solid #ccc;"><td><b>合計</b></td><td style="text-align: right;"><b>{metrics['total_time']:.3f}s</b></td></tr>
    </table>
    
    <h3>🔤 トークン使用量</h3>
    <table style="width: 100%;">
        <tr><td>入力</td><td style="text-align: right;"><b>{metrics['tokens']['input_tokens']}</b></td></tr>
        <tr><td>出力</td><td style="text-align: right;"><b>{metrics['tokens']['output_tokens']}</b></td></tr>
        <tr style="border-top: 1px solid #ccc;"><td><b>合計</b></td><td style="text-align: right;"><b>{metrics['tokens']['total_tokens']}</b></td></tr>
    </table>
    
    <h3>📚 検索結果</h3>
    <p>見つかったドキュメント数: <b>{metrics['num_search_results']}</b></p>
    </div>
    """
    return html

def format_sources_display(sources):
    """出典情報をフォーマット表示"""
    if not sources:
        return "<p>出典なし</p>"
    
    html = "<ul>"
    for source_info in sources:
        source = source_info.get('source', 'Unknown')
        folder = source_info.get('folder', '')
        page = source_info.get('page', '?')
        
        if folder and folder != '.':
            source_display = f"{folder}/{source}"
        else:
            source_display = source
        
        html += f"<li>{source_display} (p.{page})</li>"
    html += "</ul>"
    return html

def format_search_results_display(search_results):
    """検索結果の詳細をフォーマット表示"""
    html = "<div style='background-color: #fffacd; padding: 10px; border-radius: 5px; margin: 10px 0;'>"
    for i, result in enumerate(search_results, 1):
        metadata = result.get('metadata', {})
        source = metadata.get('source', 'Unknown')
        folder = metadata.get('folder', '')
        page = metadata.get('page', '?')
        distance = result.get('distance', 0)
        text = result.get('text', '')[:150]
        
        if folder and folder != '.':
            source_display = f"{folder}/{source}"
        else:
            source_display = source
        
        html += f"""<h4>[{i}] {source_display} (p.{page})</h4>
        <p style="font-size: 0.9em; color: #666;">スコア: {distance:.3f}</p>
        <p style="margin: 5px 0; font-size: 0.95em;">{text}...</p>
        <hr style="margin: 10px 0;">
        """
    html += "</div>"
    return html

print("準備完了。下記の関数でクエリを実行してください。")

# 3.実験用関数の定義

## 3.2 [本番仕様] AIに質問する関数

In [ ]:
# ====== チャットボットに質問する関数 ======
def ask_to_chatbot(query):
    """RAGに質問し、回答・出典・処理時間・トークン・検索結果を表示する"""
    print(f"\n🔍 質問: {query}")
    print("処理中...\n")

    # Execute with detailed info
    result = pipeline.query_with_info(query, top_k=config.CHROMA_TOP_K)

    # Display results
    print("\n" + "="*60)
    print("📋 回答")
    print("="*60)
    print(result['answer'])

    print("\n" + "="*60)
    print("📌 出典")
    print("="*60)
    if result['sources']:
        for source_info in result['sources']:
            source = source_info.get('source', 'Unknown')
            folder = source_info.get('folder', '')
            page = source_info.get('page', '?')

            if folder and folder != '.':
                source_display = f"{folder}/{source}"
            else:
                source_display = source

            print(f"  - {source_display} (p.{page})")
    else:
        print("  なし")

    print("\n" + "="*60)
    print("⏱️ 処理時間（秒）")
    print("="*60)
    print(f"  エンベディング: {result['metrics']['embedding_time']:.3f}s")
    print(f"  検索:          {result['metrics']['search_time']:.3f}s")
    print(f"  生成:          {result['metrics']['generation_time']:.3f}s")
    print(f"  ---")
    print(f"  合計:          {result['metrics']['total_time']:.3f}s")

    print("\n" + "="*60)
    print("🔤 トークン使用量")
    print("="*60)
    tokens = result['metrics']['tokens']
    print(f"  入力:  {tokens['input_tokens']}")
    print(f"  出力:  {tokens['output_tokens']}")
    print(f"  ---")
    print(f"  合計:  {tokens['total_tokens']}")

    print("\n" + "="*60)
    print("📚 検索結果（詳細）")
    print("="*60)
    for i, result_item in enumerate(result['search_results'], 1):
        metadata = result_item.get('metadata', {})
        source = metadata.get('source', 'Unknown')
        folder = metadata.get('folder', '')
        page = metadata.get('page', '?')
        distance = result_item.get('distance', 0)
        text = result_item.get('text', '')[:100]

        if folder and folder != '.':
            source_display = f"{folder}/{source}"
        else:
            source_display = source

        print(f"\n  [{i}] {source_display} (p.{page})")
        print(f"      スコア: {distance:.3f}")
        print(f"      テキスト: {text}...")

    return result


# ====== 使い方: 質問文を変えて実行 ======
# ask_to_chatbot("鉈を使うときの注意点は？")

### 3.2 [テスト用] 類似度検索のみ

In [ ]:
# ====== 検索のみ（embedding + ベクトル検索）を試す関数 ======
def serche_query(query):
    """Geminiを使わず、検索結果（スコアと取得チャンク）だけを表示する"""
    import time

    print(f"🔍 検索: {query}")
    print("処理中...\n")

    # Embedding
    query_embedding = embedder.embed_single(query)

    # Search
    start = time.time()
    results = retriever.search(query_embedding, top_k=config.CHROMA_TOP_K)
    search_time = time.time() - start

    print(f"\n検索完了: {search_time:.3f}s")
    print(f"見つかったドキュメント: {len(results)}")
    print("\n" + "="*60)

    for i, result in enumerate(results, 1):
        metadata = result.get('metadata', {})
        source = metadata.get('source', 'Unknown')
        folder = metadata.get('folder', '')
        page = metadata.get('page', '?')
        distance = result.get('distance', 0)
        text = result.get('text', '')

        if folder and folder != '.':
            source_display = f"{folder}/{source}"
        else:
            source_display = source

        print(f"\n[{i}] {source_display} (p.{page})")
        print(f"    スコア: {distance:.3f}")
        print(f"    内容:")
        for line in text.split('\n'):
            if line.strip():
                print(f"    {line}")

    return results


# ====== 使い方: 検索文を変えて実行 ======
# serche_query("肝試しで怖がらせるような仕掛けの例は？")

In [ ]:
ask_to_chatbot("総リーのしごとをくわしく教えて？")
# serche_query("総リーの仕事は？")